# 07. 台湾の外れ値の統計的評価とICP基準年（2021年）での確認

新しいデータは取得せず、既存の`03`の出力と`data/raw/pwt110_raw.xlsx`から計算する。
- 2023年横断面（40か国・地域）での台湾の外部スチューデント化残差、Cook's D、1か国除外（台湾を除く39か国・地域）による95%個別予測区間
- PWT 11.0の`i_cig`（物価データの性格：Benchmark／Extrapolated等）の確認
- ICP基準年である2021年の横断面での台湾・日本の残差と順位

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR = BASE_DIR / "data" / "raw"
TABLES_DIR = BASE_DIR / "output" / "tables"
cs = pd.read_csv(PROCESSED_DIR / "penn_regression_2023_highincome.csv")

In [2]:
# 2023年横断面：影響診断と台湾を除いた予測区間
x, y = cs.log_x.to_numpy(), cs.log_y.to_numpy()
n, p = len(x), 2
X = np.c_[np.ones(n), x]
b = np.linalg.lstsq(X, y, rcond=None)[0]
e = y - X @ b
h = np.diag(X @ np.linalg.inv(X.T @ X) @ X.T)
s2 = e @ e / (n - p)
s2_del = (e @ e - e**2 / (1 - h)) / (n - p - 1)
t_ext = e / np.sqrt(s2_del * (1 - h))
cook = e**2 / (p * s2) * h / (1 - h)**2
diag = pd.DataFrame({"countrycode": cs.countrycode, "ext_studentized": t_ext, "cooks_d": cook, "leverage": h})
diag["p_two_sided"] = 2 * stats.t.sf(np.abs(t_ext), n - p - 1)
diag["p_bonferroni"] = np.minimum(1, diag.p_two_sided * n)
print(diag.reindex(diag.ext_studentized.abs().sort_values(ascending=False).index).head(5).round(4).to_string(index=False))

i = int(np.where(cs.countrycode == "TWN")[0][0])
keep = np.arange(n) != i
bm = np.linalg.lstsq(X[keep], y[keep], rcond=None)[0]
em = y[keep] - X[keep] @ bm
sm = np.sqrt(em @ em / (n - 1 - p))
x0 = X[i]
pred = x0 @ bm
se_pred = sm * np.sqrt(1 + x0 @ np.linalg.inv(X[keep].T @ X[keep]) @ x0)
tc = stats.t.ppf(0.975, n - 1 - p)
loo = pd.Series({
    "n_without_TWN": n - 1, "slope_without_TWN": bm[1],
    "TWN_actual_relp": np.exp(y[i]), "TWN_predicted_relp": np.exp(pred),
    "PI95_lower": np.exp(pred - tc * se_pred), "PI95_upper": np.exp(pred + tc * se_pred),
})
print(loo.round(3).to_string())
out = diag[diag.countrycode == "TWN"].assign(**loo.to_dict())
out.to_csv(TABLES_DIR / "taiwan_outlier_diagnostics_2023.csv", index=False)

countrycode  ext_studentized  cooks_d  leverage  p_two_sided  p_bonferroni
        TWN          -4.3136   0.3426    0.0511       0.0001        0.0046
        TUR          -2.2454   0.1849    0.0751       0.0308        1.0000
        ISR           2.0106   0.0480    0.0250       0.0517        1.0000
        MYS          -1.9689   0.1829    0.0921       0.0565        1.0000
        NZL           1.3925   0.0250    0.0257       0.1721        1.0000
n_without_TWN         39.000
slope_without_TWN      1.088
TWN_actual_relp        0.443
TWN_predicted_relp     0.958
PI95_lower             0.667
PI95_upper             1.376


In [3]:
# PWT 11.0 の i_cig（2023年標本の物価データの性格）
raw = pd.read_excel(RAW_DIR / "pwt110_raw.xlsx", sheet_name="Data", usecols=["countrycode", "year", "i_cig"])
sub = raw[raw.countrycode.isin(cs.countrycode)]
icig = sub[sub.year.isin([2021, 2022, 2023])].groupby(["year", "i_cig"]).size().rename("n").reset_index()
print(icig.to_string(index=False))
print(raw[raw.countrycode.isin(["JPN", "TWN", "USA"]) & raw.year.isin([2021, 2023])].to_string(index=False))
icig.to_csv(TABLES_DIR / "pwt_icig_sample_2021_2023.csv", index=False)

 year        i_cig  n
 2021    Benchmark 40
 2022 Extrapolated 40
 2023 Extrapolated 40
countrycode  year        i_cig
        JPN  2021    Benchmark
        JPN  2023 Extrapolated
        TWN  2021    Benchmark
        TWN  2023 Extrapolated
        USA  2021    Benchmark
        USA  2023 Extrapolated


In [4]:
# ICP基準年（2021年）の横断面：年別回帰（03の出力）から
by = pd.read_csv(PROCESSED_DIR / "penn_regression_by_year.csv")
y21 = by[by.year == 2021].sort_values("resid_pct").reset_index(drop=True)
y21["rank_low"] = np.arange(1, len(y21) + 1)
res21 = y21[y21.countrycode.isin(["TWN", "KOR", "JPN"])][["countrycode", "resid_pct", "rank_low", "fit_slope"]]
res21 = res21.assign(n=len(y21))
print(res21.round(3).to_string(index=False))
res21.to_csv(TABLES_DIR / "penn_regression_2021_benchmark_year.csv", index=False)

countrycode  resid_pct  rank_low  fit_slope  n
        TWN    -46.193         1      0.876 37
        KOR     -6.284        14      0.876 37
        JPN     40.473        36      0.876 37
